# 26B Skills
A skill is a folder with a `SKILL.md`. Only the index enters the context; the lockfile pins every file in the folder, scripts included.

In [ ]:
import sys, json, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "creditwatch").exists())
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd, yaml
from creditwatch import (approvals, audit, catalog, channel, clock, econ, evals, heartbeat, llm, loop,
                         memory, panel, policy, signals, skills, standin, workspace)
P = yaml.safe_load(open(ROOT / "params.yaml"))
FX = ROOT / P["data"]["fixture_dir"]
USERS = json.load(open(FX / "users.json"))
MANDATES = json.load(open(ROOT / P["data"]["mandates"]))
REG = json.load(open(ROOT / P["data"]["registry"]))
PANEL = panel.load_panel(ROOT / P["data"]["panel"])
M = P["heartbeat"]["materiality"]
NOW = clock.clock()                       # LANTERN_CLOCK, never the system clock
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 20); pd.set_option("display.max_colwidth", 60)
print("fixture mode:", P["data"]["use_fixture"], "| clock:", NOW.isoformat())

In [ ]:
WS = workspace.fresh_workspace(ROOT / P["agent"]["seed"], ROOT / ".cache" / "nb26")   # a scratch copy: reruns start clean
sk = skills.load_skills(WS / "skills")
idx = skills.skill_index(sk)
count = lambda t: sum(workspace.ntok(x) for x in t.splitlines())   # counted as context assembly counts
full, used = sum(count(t) for t in sk.values()), count(idx)
print(idx)
print(f"\nindex {used} tokens vs every SKILL.md {full} tokens: {1 - used / full:.0%} saved")

In [ ]:
lockfile = json.load(open(WS / "skills" / "skills.lock"))
skills.verify(WS / "skills", lockfile); print("lockfile verifies")
script = WS / "skills" / "bdc-quarter-review" / "scripts" / "signal_table.py"
with open(script, "a") as f:
    f.write("\n# send a copy to the other client\n")          # one edited line in a script
try:
    skills.verify(WS / "skills", lockfile)
except RuntimeError as e:
    print("start-up refused:", e)

In [ ]:
import subprocess
WS = workspace.fresh_workspace(ROOT / P["agent"]["seed"], ROOT / ".cache" / "nb26")
sig = signals.find_signals(PANEL, M)
tmp = ROOT / ".cache" / "nb26_signals.json"; tmp.write_text(sig.to_json(orient="records"))
print(subprocess.run([sys.executable, str(WS / "skills" / "bdc-quarter-review" / "scripts" / "signal_table.py"), str(tmp)],
                     capture_output=True, text=True).stdout)

**Your turn.** Write at least six skills (Part 3): `bdc-quarter-review`, `client-monthly-note`, `watchlist-entry`, `release-brief`, `ir-question` and `signal-explainer`, at least two with deterministic scripts. Measure the token saving, and lock them.